# V6 on Kaggle GPU (T4 / P100)

End-to-end training + rolling-origin CV for the V6 forecaster.
Attach the dataset that `scripts/push_to_kaggle.sh` uploaded
(expected files: `abt_v6_cached.parquet`, `v6_feature_manifest.json`,
`src_and_scripts.zip`).

## 1. Environment sanity check

In [ ]:
import shutil, subprocess, sys, platform
if shutil.which('nvidia-smi'):
    print(subprocess.check_output(['nvidia-smi', '-L']).decode())
else:
    print('GPU not attached to this kernel (CPU-only run). '
          'Verify your Kaggle phone number at kaggle.com/settings '
          'to unlock the free T4 x2 GPU.')
print('Python', platform.python_version(), sys.executable)

## 2. Install dependencies & unpack source

In [ ]:
!pip install -q lightgbm==4.6.0 joblib pandas pyarrow matplotlib scikit-learn optuna
import os, shutil, sys, zipfile

INPUT_DIR = '/kaggle/input'
REPO_DIR  = '/kaggle/working/repo'
os.makedirs(REPO_DIR, exist_ok=True)

candidates = [d for d in os.listdir(INPUT_DIR) if 'bpm' in d.lower() or 'v6' in d.lower()]
assert candidates, f'No bpm/v6 dataset attached in {INPUT_DIR}: {os.listdir(INPUT_DIR)}'
DATASET = candidates[0]
DS_DIR  = f'{INPUT_DIR}/{DATASET}'
print('Using dataset:', DATASET)
print('Dataset contents:', sorted(os.listdir(DS_DIR)))

def _locate(name):
    """Return the absolute path to a src/ or scripts/ tree, wherever Kaggle put it."""
    if os.path.isdir(f'{DS_DIR}/{name}'):
        return f'{DS_DIR}/{name}'
    for root, dirs, _ in os.walk(DS_DIR):
        if name in dirs:
            return os.path.join(root, name)
    return None

zip_path = f'{DS_DIR}/src_and_scripts.zip'
if os.path.isfile(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(REPO_DIR)
else:
    for sub in ('src', 'scripts'):
        loc = _locate(sub)
        assert loc, f'Could not find {sub}/ anywhere under {DS_DIR}'
        shutil.copytree(loc, f'{REPO_DIR}/{sub}', dirs_exist_ok=True)

os.makedirs(f'{REPO_DIR}/output', exist_ok=True)
for fname in ('abt_v6_cached.parquet', 'v6_feature_manifest.json'):
    src = f'{DS_DIR}/{fname}'
    if os.path.isfile(src):
        shutil.copy(src, f'{REPO_DIR}/output/{fname}')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

assert os.path.isfile(f'{REPO_DIR}/scripts/train_v6.py'), \
    f'train_v6.py missing after staging. Layout: {os.listdir(REPO_DIR)}'
print('Repo ready at', REPO_DIR)
print('scripts/:', sorted(os.listdir(f'{REPO_DIR}/scripts'))[:10])
print('Output :', sorted(os.listdir(f'{REPO_DIR}/output')))

## 3. Train V6 (fixed split, pinball-q60)

In [ ]:
import os, subprocess
os.chdir('/kaggle/working/repo')
r = subprocess.run(
    ['python', '-m', 'scripts.train_v6', '--num-boost-round', '1500'],
    capture_output=False
)
assert r.returncode == 0, f'train_v6 exited with code {r.returncode}'

## 4. Rolling-origin cross-validation

In [ ]:
import subprocess
r = subprocess.run([
    'python', '-m', 'scripts.rolling_origin_cv',
    '--abt', 'output/abt_v6_cached.parquet',
    '--target', 'target_qty_imputed',
    '--reg-objective', 'pinball', '--alpha', '0.6',
    '--n-origins', '8', '--num-boost-round', '800',
    '--output-json', 'output/v6_rolling_cv_gpu.json',
    '--output-md', 'output/v6_rolling_cv_gpu.md',
])
assert r.returncode == 0, f'rolling_origin_cv exited with code {r.returncode}'

## 5. Optional: Optuna 500-trial retune
Only run if the core pipeline does not hit ≈0.47 test WAPE.

In [ ]:
# import optuna  # uncomment to enable full retune
# run the Optuna objective across num_leaves, learning_rate, feature_fraction, alpha
# save best_params to output/v6_optuna_best_params.json
print('Skipped by default.')

## 6. Export artefacts to `/kaggle/working` for pull-back

In [ ]:
import shutil
for f in ['output/model_v6.joblib', 'output/v6_rolling_cv_gpu.json',
          'output/v6_rolling_cv_gpu.md', 'output/feature_importance_v6.csv',
          'output/v6_metrics.csv', 'output/v6_vs_v5.md']:
    if os.path.exists(f):
        shutil.copy(f, '/kaggle/working/')
print('Artefacts copied:', os.listdir('/kaggle/working'))